# 01. Download DECam Image
Written by Kiyoaki Okudaira<br>
*University of Washington / Kyushu University / IAU CPS SatHub<br>
(kiyoaki@uw.edu or okudaira.kiyoaki.528@s.kyushu-u.ac.jp)<br>
<br>
This code is written for ASTR 499 undergraduate research with Dr. Meredith.<br>
Download DECam images from noirlab API<br>
<br>
**History**<br>
coding 2025-11-09 : 1st coding<br>
update 2025-11-16 : adapted for Middle Earth<br>
update 2025-11-23 : adapted for sp library<br>
update 2026-05-02 : output filename changed & parallel processing

### Initial Settings
**Libraries**

In [ ]:
# Standard libraries
from os import path
import numpy as np

# Public libraries
from astropy.table import Table, Column
from tqdm.notebook import tqdm

# Custom libraries
import satphotometry as sp

**Import file settings**

In [ ]:
# project directory
PATH_project = '/astro/store/shire/kiyoaki/ASTR499/'

PATH_input   = PATH_project + 'input/'
PATH_output  = PATH_project + 'output/'

PATH_image   = PATH_input  + 'fits_data/'
PATH_session = PATH_output + 'session/'

# ORIGINAL streak list from Alex
fname_streak_list = 'streaks_augmented_20230817.csv'
PATH_streak_list  = fname_streak_list + 'decam_streak_list/' + fname_streak_list

### Read DECam streak list

In [ ]:
data = Table.read(PATH_output+path.splitext(path.basename(fname_streak_list))[0]+'_00_masked_by_year.csv')

data.add_column(Column(np.arange(1,len(data)+1,1), name='streakID'), index=0)
data.write(PATH_output+path.splitext(path.basename(fname_streak_list))[0]+'_00_masked_by_year.csv',overwrite=True)

data = data.group_by("archive_filename")

data

### Process
**Single processing**

In [ ]:
# for group in tqdm(data.groups):
#     for i in range(0,len(group)):
#         row = group[i]

#         basename = path.basename(row["archive_filename"])
#         md5sum = row["md5sum"]
#         expnum = row["expnum"]
#         ccdnum = row["ccdnum"]

#         save_path = PATH_image + path.splitext(path.splitext(basename)[0])[0] + "_CCD_{0}".format(ccdnum) + path.splitext(path.splitext(basename)[0])[1] + path.splitext(basename)[1]

#         if path.exists(save_path) is not True:
#             sp.noirlab.retrieve_fits_nocash(md5sum,save_path,ccdnum)

**Parallel processing**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_row(row, download_path):
    basename = path.basename(row["archive_filename"])
    md5sum = row["md5sum"]
    expnum = row["expnum"]
    ccdnum = row["ccdnum"]

    save_path = (
        download_path
        + path.splitext(path.splitext(basename)[0])[0]
        + f"_CCD_{ccdnum}"
        + path.splitext(path.splitext(basename)[0])[1]
        + path.splitext(basename)[1]
    )

    if not path.exists(save_path):
        sp.noirlab.retrieve_fits_nocash(md5sum, save_path, ccdnum)

    # hdu = fits.open(save_path)

    return {
        "expnum": expnum,
        "ccdnum": ccdnum,
        "save_path": save_path
    }

futures = []
results = []

max_workers = 3

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    for group in data.groups:
        for i in range(len(group)):
            row = group[i]
            futures.append(executor.submit(process_row, row, PATH_image))

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading/FITS opening"):
        results.append(future.result())